# New Bedford, MA — Verified CSO Discharges

Source: MassDEP CSO Data Portal, filtered to `Verified Data Report` events (measured volume, duration, rainfall).

In [1]:
import pandas as pd
import hvplot.pandas  # noqa

In [2]:
df = pd.read_csv(
    "new_bedford_cso_discharges_verified.csv",
    parse_dates=["discharge_date", "submitted_date"],
)
df["volume_discharged_gallons"] = pd.to_numeric(df["volume_discharged_gallons"], errors="coerce")

# Two rows have physically impossible rainfall_in values (400,252 in and 62,060 in),
# apparent data-entry errors in the source spreadsheet — drop them everywhere.
df = df[df["rainfall_in"] <= 20].copy()

df.head()

,outfall_id,discharge_date,discharge_time,event_type,duration,volume_discharged_gallons,rainfall_in,location,water_body,incident_number,submitted_date
0,NEW003,2022-07-06,12:30 AM,CSO - UnTreated,12h 33m,230000,0.04,COVE RD & PADNARAM AVE,CLARK COVE,22000350,2022-08-29 07:08:52
1,NEW018,2022-07-07,2:19 PM,CSO - UnTreated,1h 36m,100000,0.05,COVE ST AND E. RODNEY FRENCH BLVD,OUTER NEW BEDFORD HARBOR,22000351,2022-08-29 07:12:20
2,NEW022,2022-07-09,3:14 AM,CSO - UnTreated,6h 40m,40000,0.05,SAWYER ST AT ACUSHNET R,ACUSHNET RIVER,22000352,2022-08-29 07:14:50
3,NEW022,2022-07-12,1:15 PM,CSO - UnTreated,0h 40m,0,0.00,SAWYER ST AT ACUSHNET R,ACUSHNET RIVER,22000353,2022-08-29 07:16:57
4,NEW013,2022-07-12,6:00 PM,CSO - UnTreated,2h 0m,2080000,0.00,AQUIDNECK & RFB,OUTER NEW BEDFORD HARBOR,22000354,2022-08-29 07:19:02


In [3]:
df.shape

(1725, 11)

## Discharge volume over time, by outfall

In [4]:
df.hvplot.scatter(
    x="discharge_date",
    y="volume_discharged_gallons",
    color="outfall_id",
    hover_cols=["outfall_id", "location", "water_body", "duration", "incident_number"],
    yformatter="%.0f",
    ylabel="Volume discharged (gallons)",
    xlabel="Discharge date",
    title="New Bedford Verified CSO Discharges Over Time",
    height=450,
    responsive=True,
    legend="right",
)

:Scatter   [discharge_date]   (volume_discharged_gallons,outfall_id,location,water_body,duration,incident_number)

## Total volume discharged by outfall

In [5]:
by_outfall = (
    df.groupby("outfall_id")["volume_discharged_gallons"]
    .sum()
    .sort_values()
    .reset_index()
)

by_outfall.hvplot.barh(
    x="outfall_id",
    y="volume_discharged_gallons",
    hover_cols=["outfall_id"],
    hover_tooltips=[("Outfall", "@outfall_id"), ("Total gallons", "@volume_discharged_gallons{0,0}")],
    xlabel="Total gallons discharged (2022\u20132026)",
    ylabel="Outfall",
    title="Total Verified Discharge Volume by Outfall",
    height=500,
    responsive=True,
    color="#4e79a7",
)

:Bars   [outfall_id]   (volume_discharged_gallons)

## Monthly total volume discharged

In [6]:
monthly = (
    df.set_index("discharge_date")["volume_discharged_gallons"]
    .resample("ME")
    .sum()
)

monthly.hvplot.line(
    ylabel="Total gallons discharged",
    xlabel="Month",
    title="Monthly Total CSO Volume — New Bedford",
    height=350,
    responsive=True,
).opts(hover_mode="vline")

:Curve   [discharge_date]   (volume_discharged_gallons)

## Total discharge volume by water body — Summer 2025 (Jun–Aug)

In [7]:
summer_2025 = df[
    (df["discharge_date"] >= "2025-06-01") & (df["discharge_date"] <= "2025-08-31")
].copy()

by_water_body = (
    summer_2025.groupby("water_body")["volume_discharged_gallons"]
    .sum()
    .sort_values()
    .reset_index()
)

by_water_body.hvplot.barh(
    x="water_body",
    y="volume_discharged_gallons",
    hover_cols=["water_body"],
    hover_tooltips=[("Water body", "@water_body"), ("Total gallons", "@volume_discharged_gallons{0,0}")],
    xlabel="Total gallons discharged — Summer 2025",
    ylabel="Water body",
    title="Total CSO Volume by Water Body — Summer 2025",
    height=350,
    responsive=True,
    color="#e15759",
)

:Bars   [water_body]   (volume_discharged_gallons)

## Individual events by water body — Summer 2025 (Jun–Aug)

One panel per water body; bar height is the volume discharged for that specific event.

In [8]:
summer_2025["event_label"] = (
    summer_2025["discharge_date"].dt.strftime("%b %d") + " " + summer_2025["discharge_time"]
)
summer_2025 = summer_2025.sort_values(["water_body", "discharge_date"])

summer_2025.hvplot.bar(
    x="event_label",
    y="volume_discharged_gallons",
    by="water_body",
    subplots=True,
    shared_axes=False,
    legend=False,
    rot=45,
    hover_cols=["outfall_id", "location", "duration", "incident_number"],
    hover_tooltips=[
        ("Event", "@event_label"),
        ("Gallons", "@volume_discharged_gallons{0,0}"),
        ("Outfall", "@outfall_id"),
        ("Location", "@location"),
        ("Duration", "@duration"),
    ],
    xlabel="",
    ylabel="Gallons discharged",
    color="#4e79a7",
    height=350,
    width=450,
    fontscale=0.9,
).cols(2)

:NdLayout   [water_body]
   :Bars   [event_label]   (volume_discharged_gallons,outfall_id,location,duration,incident_number)

## Discharge volume vs. rainfall, by water body

One scatter panel per water body (all years), with a linear regression line fit to volume vs. rainfall.

In [9]:
import numpy as np
import holoviews as hv

def scatter_with_trend(sub, water_body):
    x = sub["rainfall_in"].to_numpy()
    y = sub["volume_discharged_gallons"].to_numpy()

    scatter = sub.hvplot.scatter(
        x="rainfall_in",
        y="volume_discharged_gallons",
        hover_cols=["outfall_id", "discharge_date", "location"],
        xlabel="Rainfall (in)",
        ylabel="Gallons discharged",
        color="#4e79a7",
        size=40,
        alpha=0.7,
        height=350,
        responsive=True,
    )

    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    trend = hv.Curve((xs, slope * xs + intercept), label=f"y = {slope:,.0f}x + {intercept:,.0f}").opts(
        color="#e15759", line_width=2
    )

    return (scatter * trend).opts(title=water_body, legend_position="top_left")

panels = {
    water_body: scatter_with_trend(df[df["water_body"] == water_body], water_body)
    for water_body in sorted(df["water_body"].unique())
}

hv.NdLayout(panels).cols(2)

:NdLayout   [Default]
   :Overlay
      .Scatter.I                                                             :Scatter   [rainfall_in]   (volume_discharged_gallons,outfall_id,discharge_date,location)
      .Curve.Y_equals_1_comma_534_comma_896x_plus_hyphen_minus_472_comma_307 :Curve   [x]   (y)